# SD2.1 + Pick-a-Pic Experiment (Standard vs. Z-Sampling)

This notebook provides a self-contained demonstration of the SD2.1 Pick-a-Pic experiment. It reuses the modular components from the `zigzag_project` package to ensure parity with the main headless scripts

In [ ]:
import os
import sys

sys.path.append(os.path.abspath('../src'))

import matplotlib.pyplot as plt

from zigzag_project.experiment_io import load_prompt_specs
from zigzag_project.models import load_sd21
from zigzag_project.zsampling import ZSamplingConfig, make_initial_latents, z_sample_sd21

## 1. Load Prompts
Load the prompts from the `pickapic_full.md`

In [ ]:
prompts = load_prompt_specs("../experiments/prompts/pickapic_full.md", prefix="pickapic")
test_prompt = prompts[0]

print(f"Loaded {len(prompts)} prompts")
print(f"Testing with Prompt 0: '{test_prompt.prompt}'")

## 2. Load Model
Load Stable Diffusion 2.1 using the custom loader. 
This handles setting up the DDIM and DDIMInverse schedulers required for Z-Sampling.

In [ ]:
handle = load_sd21()
print(f"Model loaded on {handle.device} with {handle.dtype}")

## 3. Configuration & Latents
Set the hyperparameters based on the paper's SD2.1 specifications.

In [ ]:
seed = 42
size = 768
steps = 50
cfg = 7.5
inv_cfg = 0.0
lambda_step = 49
t_max = 1

latents = make_initial_latents(
    seed=seed,
    height=size,
    width=size,
    dtype=handle.dtype,
    device=handle.device,
)

## 4. Standard Sampling

In [ ]:
print("Generating Standard Image...")
standard_image = handle.pipe(
    prompt=test_prompt.prompt,
    num_inference_steps=steps,
    guidance_scale=cfg,
    height=size,
    width=size,
    latents=latents.clone(),
).images[0]

plt.figure(figsize=(8, 8))
plt.imshow(standard_image)
plt.axis('off')
plt.title(f"Standard Sampling (CFG={cfg})")
plt.show()

## 5. Z-Sampling

In [ ]:
print("Generating Z-Sampling Image...")
config = ZSamplingConfig(
    gamma_1=cfg,
    gamma_2=inv_cfg,
    infer_step=steps,
    lambda_step=lambda_step,
    t_max=t_max,
)

zigzag_image = z_sample_sd21(
    handle,
    prompt=test_prompt.prompt,
    config=config,
    height=size,
    width=size,
    latents=latents.clone(),
).images[0]

plt.figure(figsize=(8, 8))
plt.imshow(zigzag_image)
plt.axis('off')
plt.title(f"Z-Sampling (lambda={lambda_step}, T_max={t_max})")
plt.show()

## 6. Side-by-Side Comparison
View the generated images side-by-side to observe the qualitative difference (based on sampling)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].imshow(standard_image)
axes[0].set_title(f"Standard Sampling (CFG={cfg})")
axes[0].axis('off')

axes[1].imshow(zigzag_image)
axes[1].set_title(f"Z-Sampling (lambda={lambda_step}, T_max={t_max})")
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 7. Memory Management
Not exactly sure how to do memory management, this was the best suggestion

In [ ]:
import gc

import torch


def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()
    print("Memory freed.")

free_memory()

## 8. HPSv2 Evaluation
Evaluate the prompt alignment using HPSv2

(Ensure you have installed `hpsv2` via `pip install hpsv2` before running this block)

In [ ]:
try:
    import os
    import tempfile

    import hpsv2

    # Temporarily save images to disk to score them using HPSv2
    with tempfile.TemporaryDirectory() as tmpdir:
        std_path = os.path.join(tmpdir, "std.png")
        zz_path = os.path.join(tmpdir, "zz.png")
        standard_image.save(std_path)
        zigzag_image.save(zz_path)

        # Score the images against the prompt
        print(f"Evaluating against prompt: {test_prompt.prompt}")
        std_score = hpsv2.score([std_path], test_prompt.prompt, hps_version="v2.1")[0]
        zz_score = hpsv2.score([zz_path], test_prompt.prompt, hps_version="v2.1")[0]

        print(f"Standard Sampling HPSv2 Score: {std_score:.4f}")
        print(f"Z-Sampling HPSv2 Score: {zz_score:.4f}")

        if zz_score > std_score:
            print("Result: Z-Sampling produced higher prompt alignment.")
        else:
            print("Result: Standard Sampling produced higher prompt alignment.")

except ImportError:
    print("HPSv2 is not installed in the current environment.")
    print("To run evaluation on a full experiment manifest, you can use the CLI:")
    print(
        "!PYTHONPATH=src .venv/bin/python scripts/score_hpsv2_pairs.py "
        "--manifest path/to/manifest.jsonl --out path/to/scores.jsonl"
    )
